In [ ]:
#FROM compare_complex_walks.ipynb 

#make akita predictions for each walk and plot 
def akita_pred(mcool):
    i=mcool.replace('.fa.mcool', '.fa')
    print(i)
    bed_name=i[:-3] +'.bed'
    name_parts = i.rsplit('_', 2)

    gencode_coord=gene_annot[gene_annot.Gene == name_parts[1]]    
    gene_len=abs(gencode_coord.end_gene-gencode_coord.start_gene)
    gene_mid_coord=np.floor(gencode_coord.start_gene + (gene_len/2))
    #print('gene mid coord', gene_mid_coord)
    #print('gencode coord', gencode_coord)
    
    #read in bed file and figure out where to center the file 
    bed=pd.read_csv(data_dir+ 'new_fasta/walks.bed/' + bed_name, sep='\t', header=None)
    #rename columns
    bed.columns = ['chr', 'start', 'end', 'strand', '4', '5']
    bed['chr'] = bed['chr'].astype('str') 
    #get spans of each walk 
    bed['span']=abs(bed['end']-bed['start']) 
    
    #print('bed span', np.sum(bed['span']))


    #get the row where gene_mid_coord 
    #print(gencode_coord['CHROM'])
    condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gene_mid_coord)) & (int(gene_mid_coord) <= bed['end']) ) 
    #in cases where the middle of the gene shows up more than once, just takes first instance 
    row_midgene = (condition.index[condition].tolist())[0]
    #print('row_midgene index', row_midgene)
                

    #get span for the entire walk up to the middle of the gene 
    bed_row=bed.iloc[row_midgene, :]
    #print('bed_Row', bed_row)
    #print('gene row span', np.sum(bed_row['span']))
    #print('gene_mid_Coord', gene_mid_coord)
    if row_midgene>0:
        bed_begin=bed.iloc[0:row_midgene, :]
        #print('bed_begin', bed_begin)
        first_half_span=(np.sum(bed_begin['span'])) + int(gene_mid_coord-bed_row['start'])
        bed_before=pd.concat([bed_begin, bed_row])

    else:
        first_half_span=int(gene_mid_coord-bed_row['start'])
        bed_before=bed_row
    #print('bed_before', bed_row)
            
    #then, get span for entire walk from middle of gene to end 
    if row_midgene<(len(bed)-1):
        bed_end=bed.iloc[(row_midgene+1):len(bed), :]
        #print('bed_end', bed_end)
        second_half_span=(np.sum(bed_end['span'])) + int(bed_row['end']-gene_mid_coord)
    else:
        second_half_span=int(bed_row['end']-gene_mid_coord)

        
    if 1==1:   
        #read in fasta file 
        seq=read_fasta(data_dir + 'new_fasta/walks.fasta/' + i)

     
    
        #define ALT from fasta sequences given to us       
        #check length of walk start to middle of gene, must be at lesat half of akita_seq_len
        mid_seq_idx=0
        gene_bin_start=0
        gene_bin_end=0
        seq_start=0
        seq_end=0
        gene_half=round(gene_len/2)
        
        #at least right now, no sequences need to be padded on both left and right so this works.
        #in future, if that's the case, we will need to change this
        if first_half_span<=(utils.half_patch_size):
            print('starting from left most sequence')
            
            seq_start=0
            seq_end=utils.MB
            ALT_seq=seq[0:utils.MB]
            
            gene_bin_start=utils.get_bin(first_half_span-gene_half)
            gene_bin_end=utils.get_bin(first_half_span+gene_half)
          
        
            #get REF POS 
            #POS = int(gene_mid_coord-utils.half_patch_size) + (utils.half_patch_size-first_half_span)
            POS = int(gene_mid_coord-first_half_span)

        
           
        elif second_half_span<=(utils.half_patch_size):          
            print('ending at rightmost sequence ')
            
            seq_start=(len(seq)-utils.MB)
            seq_end=len(seq)       
            ALT_seq=seq[seq_start:seq_end]
            
            gene_bin_start=utils.get_bin((utils.MB-second_half_span-gene_half))
            gene_bin_end=utils.get_bin((utils.MB-second_half_span+gene_half))

                    
            #get REF POS 
            POS = int(gene_mid_coord-utils.half_patch_size) - (utils.half_patch_size-second_half_span)

            
        else:  
            seq_start=int(first_half_span-(utils.half_patch_size))
            seq_end=int(seq_start+utils.MB)  
            print(seq_start)
            print(seq_end)
            print(len(seq))
            ALT_seq=seq[seq_start:seq_end]

            gene_bin_start=utils.get_bin(utils.half_patch_size - round(gene_len/2))
            gene_bin_end=utils.get_bin(utils.half_patch_size + round(gene_len/2))
            
                    
            #get REF POS 
            POS = int(gene_mid_coord-utils.half_patch_size)
            
            
            
        #get REF                   
        #define start and end coordinates of reference matrices
        #currently, just centered around the middle of the gene of interest        
        CHR = gencode_coord.CHROM.values[0]
        END= int(POS+utils.seq_length)
        REF_seq = fasta_hg38.fetch(CHR, POS, END).upper()
            
        ####################
        
        assert len(REF_seq) == utils.seq_length, 'REF_seq is not right length'
        assert len(ALT_seq) == utils.seq_length, 'ALT_seq is not right length'
        
#         print('gene bin start', gene_bin_start)
#         print('gene bin end', gene_bin_end)
        
#         print('seq start', seq_start)
#         print('seq end', seq_end)
#         print('seq len', len(seq))

    
        #this is where akita is actually run, output vector size is (99681,)
        REF_vector, ALT_vector = utils.vector_from_seq(REF_seq), utils.vector_from_seq(ALT_seq)
        REF_mat, ALT_mat = utils.mat_from_vector(REF_vector), utils.mat_from_vector(ALT_vector)


        ALT_label=i[:-3]
        REF_label='{}:{}-{}'.format(CHR, POS, END)
        return(REF_mat, ALT_mat, REF_label, ALT_label, 
               gene_bin_start, gene_bin_end, gene_len, 
               CHR, seq_start, seq_end)


#         plot_complex_pred(ALT_mat, REF_mat, ALT_label, REF_label, gene_len)
            
        
    else:
        print('error!!!! in file ', i )

In [ ]:
#make akita predictions for each walk and plot 
def pred_akita_genecenter_COSMIC(fa_dir, fa_name, patient_num):
    bed_name=fa_name.replace('fa', 'bed')
    name_parts = bed_name.rsplit('_', 2)

    gencode_coord=gene_annot[gene_annot.Gene == name_parts[1]]    
    gene_len=abs(gencode_coord.end_gene-gencode_coord.start_gene)
    gene_mid_coord=np.floor(gencode_coord.start_gene + (gene_len/2))
    #print(gencode_coord)
    
    if 1==1:

        #read in bed file and figure out where to center the file 
        bed=pd.read_csv(f'{patient_data_dir}walks_bed/' + bed_name, sep='\t', header=None)
        bed.columns = ['chr', 'start', 'end', 'strand', 'cn', '5']
        bed['chr'] = bed['chr'].astype('str') 
        bed['span']=abs(bed['end']-bed['start']) 
        bed['span_cumsum'] = bed['span'].cumsum()
    
        walks_BED=BedTool.bed(cosmic_subset)

        #print(bed)
        
        #overlap walks bed with gene list 
        genes_in_map_BED = walks_BED.intersect(cosmic_BED, wa = True, wb = True)
        print(genes_in_map_BED)

        
        if genes_in_map_BED == '':
            genes_in_map = pd.DataFrame(columns=['Start', 'End', 'Gene', 'width'])

        else:
            genes_in_map = (genes_in_map_BED
                        .to_dataframe()
                        .rename(columns = {'score': 'Start', 'strand':'End', 'thickStart':'Gene'})
                        [['Start', 'End', 'Gene']])
        
        
        
        
        
        
        #get the row where gene_mid_coord 
        start_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.start_gene)) & (int(gencode_coord.start_gene) <= bed['end']) ) 
        row_gene_start_list = start_condition.index[start_condition].tolist()
            
        end_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.end_gene)) & (int(gencode_coord.end_gene) <= bed['end']) ) 
        row_gene_end_list = end_condition.index[end_condition].tolist()
        #print('row_gene_start_list', row_gene_start_list)

       

    
    # read in fasta file 
    seq=read_fasta(f'{fa_dir}{fa_name}')
            
    
    figures=[]
    for i in range (0, len(bed)-1):
    #for i in range(0,4):
        #print('row ', i)
        
        try:
        
            #get first, last rows, and subsets of bed before and after bkpt 
            first_row=bed.iloc[0,:]
            last_row=bed.iloc[(len(bed)-1), :]
            left_bkpt=bed.iloc[i, :]
            right_bkpt=bed.iloc[i+1,:]
            bed_to_bkpt=bed.iloc[0:(i+1),:]
        
            #get relevant lengths
            total_span=np.sum(bed['span'])
            len_left_to_bkpt=np.sum(bed_to_bkpt['span'])
            len_bkpt_to_right=total_span-len_left_to_bkpt
        
        
        
            # left sequence 
            if len_left_to_bkpt>=utils.half_patch_size:
                ALT_left=seq[(len_left_to_bkpt - utils.half_patch_size ):len_left_to_bkpt]
            
            else:   
                #need to pad left from reference 
                diff_len=utils.half_patch_size-len_left_to_bkpt
           
                first_row_idx=first_row.start
                ALT_fromref_left=fasta_hg38.fetch(first_row.chr, first_row_idx-diff_len, first_row_idx).upper()
            
                ALT_left=ALT_fromref_left + seq[0:len_left_to_bkpt]
            
        
            # right sequence 
            if len_bkpt_to_right>utils.half_patch_size:
                ALT_right=seq[len_left_to_bkpt: (len_left_to_bkpt + utils.half_patch_size)]
            
            else:
                #need to pad right from reference        
                diff_len=utils.half_patch_size-len_bkpt_to_right
            
                last_row_idx=last_row.end
                ALT_fromref_right=fasta_hg38.fetch(last_row.chr, last_row_idx, last_row_idx + diff_len).upper()

                ALT_right=seq[len_left_to_bkpt:total_span] + ALT_fromref_right
            
            
            ALT_seq = ALT_left + ALT_right

            
            ########################

            #we want to get indices of the breakpoints, if others are in the window 

            #get index for where the akita window starts 
            left_start=len_left_to_bkpt-utils.half_patch_size
            #print(left_start)

            bkpt_pos_from_left_all=bed['span_cumsum'].tolist() - left_start 
            #print(bkpt_pos_from_left_all)

            #subset to those less than akita window size
            bkpt_pos_from_left=bkpt_pos_from_left_all[(bkpt_pos_from_left_all>0) & (bkpt_pos_from_left_all<utils.MB)]
            #print(bkpt_pos_from_left)
            
            #get the bin index
            bkpt_bin_list_all=[utils.get_bin(bkpt) for bkpt in bkpt_pos_from_left]
            bkpt_bin_list = [bkpt for bkpt in bkpt_bin_list_all if 0 < bkpt <= 448]


            
            ########################
            #finally, get the positions of the gene(s)
                       
            gene_start_bin_list=[]
            gene_end_bin_list=[]

            try:
                
                #check the start of the gene 
                for g in row_gene_start_list:
                    len_to_gene_start=gencode_coord.start_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene start' , len_to_gene_start)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_start+=len_before_row


                    gene_start_pos_window=len_to_gene_start - left_start 
                    if (gene_start_pos_window>0) & (gene_start_pos_window<utils.MB):
                        gene_start_bin_list.append(utils.get_bin(gene_start_pos_window))
                        
                #check the end of the gene      
                for g in row_gene_end_list:
                    len_to_gene_end=gencode_coord.end_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene end' , len_to_gene_end)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_end+=len_before_row

                    gene_end_pos_window=len_to_gene_end - left_start 
                    if (gene_end_pos_window>0) & (gene_end_pos_window<utils.MB):
                        gene_end_bin_list.append(utils.get_bin(gene_end_pos_window))

                
            except:
                print("error in gene annotation!")


            #########################
            #finally, get the REF -- for the two sequences to the left and right of the breakpoint 
            left_ref_coords=[left_bkpt.chr, left_bkpt.end-utils.half_patch_size, 
                                          left_bkpt.end+utils.half_patch_size]
            right_ref_coords=[right_bkpt.chr, right_bkpt.start-utils.half_patch_size, 
                                          right_bkpt.start+utils.half_patch_size]


            REF_seq_left=fasta_hg38.fetch(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]).upper()
            REF_seq_right=fasta_hg38.fetch(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]).upper()


            ####################

            #check sequence lengths, and then run akita 
            #print(len(REF_seq_left))
            assert len(REF_seq_left) == utils.seq_length, 'REF_seq_left is not right length'
            assert len(REF_seq_right) == utils.seq_length, 'REF_seq_right is not right length'
            assert len(ALT_seq) == utils.seq_length, 'ALT_seq is not right length'



            #this is where akita is actually run, output vector size is (99681,)
            REF_left_vec, REF_right_vec, ALT_vec = utils.vector_from_seq(REF_seq_left), utils.vector_from_seq(REF_seq_right), utils.vector_from_seq(ALT_seq)
            REF_left_mat, REF_right_mat, ALT_mat = utils.mat_from_vector(REF_left_vec), utils.mat_from_vector(REF_right_vec), utils.mat_from_vector(ALT_vec)

    
            #order is REF_left, REF_right, ALT
            mat_list=[REF_left_mat, REF_right_mat, ALT_mat]
            
            #add annotation for span between bkpts if same chr, or annot as translocation
            if left_ref_coords[0]==left_ref_coords[0]:
                span_annot=str(right_ref_coords[1] - left_ref_coords[2]) + ' bp between breakpoints'
            else:
                span_annot='interchromosomal'
            
            #get label list 
            label_list=['{}:{}-{}'.format(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]), 
                       '{}:{}-{}'.format(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]), 
                       'Left: {}:{}-{} \n Right: {}:{}-{} \n {}'.format(
                        left_bkpt.chr, left_bkpt.start, left_bkpt.end, 
                           right_bkpt.chr, right_bkpt.start, right_bkpt.end, 
                           span_annot), f'{fa_name[:-3]} \n rows {i} - {i+1}']

            #print(label_list)
            # plot!
            fig=plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list)
            figures.append(fig)
           

        
        else:
            print('error!!!! in row index', i )
            
    #print(f'saving to {fig_dir}patient_{patient_num}/{bed_name[:-4]}_breakpoints.pdf')
    #save_figures_to_pdf(f'{fig_dir}patient_{patient_num}/{bed_name[:-4]}_plot.pdf', figures)
    #plt.close('all')
